In [57]:
# shell PATH not inherited by VScode, so graphviz's `dot` wasn't found
import os, shutil
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")
print(shutil.which("dot"))

/opt/homebrew/bin/dot


In [58]:
import sys; print(sys.executable)

/Users/tomrodger/ml-foundations/.venv/bin/python


In [59]:

import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import random
print(torch.__version__)

%matplotlib inline

2.14.0


In [60]:
class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f'Value(data={self.data})'
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out 
    
    def __radd__(self, other):
        return self + other

    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)) 
        x = self.data
        out = Value(x ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += (other * self.data ** (other-1)) * out.grad
        out._backward = _backward

        return out
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += math.exp(x) * out.grad 
        out._backward = _backward

        return out 
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1) / (math.exp(2*x)+1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2 )* out.grad
        out._backward = _backward

        return out
    
    def __truediv__(self, other):
        return self * (other ** -1)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + -(other)
    
    def backward(self):

        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
    
    def __call__(self, x):
        act = sum(((wi * xi) for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]
    

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    
class MLP:
    def __init__(self, nin, nouts): 
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    

# ----

x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)
    
xs = [
    [2.0, 3.0, -1.0], 
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

# ----

for k in range(20):

    # Forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, ys))

    # Backward Pass
    for p in n.parameters(): 
        p.grad = 0.0
    loss.backward()

    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)


0 5.7523216037888085
1 4.405160720977361
2 3.5040556081582297
3 2.6043995810189364
4 1.521792180707578
5 0.741313029826802
6 0.43123628258642566
7 0.29176796151041073
8 0.21688472649848417
9 0.17097101986562185
10 0.14023623678402167
11 0.11834618726646004
12 0.1020246597617613
13 0.08942065506853884
14 0.07941455464248533
15 0.07129155055563682
16 0.06457489556159413
17 0.05893495082123734
18 0.054136803937316916
19 0.05000869547320013


In [61]:
ypred

[Value(data=0.9116190685173562),
 Value(data=-0.8719859219237458),
 Value(data=-0.8671724485322311),
 Value(data=0.9096299617747993)]